# Warm start on the IDAES CSTR

Under closed-loop control the next problem is the last one moved one
step, so the last solution moved one step is nearly its answer.
`drto.warm_start_dynamic` shifts every variable one sampling time
forward, the infinite-horizon tail included, and the next solve starts
where the last one ended. This notebook runs one loop iteration on the
IDAES saponification CSTR and reads the iteration counts.

## The first solve

The controller from the cold start, as in the cold-start example; the
solver log shows the iteration count the exponential profile buys.

In [1]:
import pyomo.environ as pyo

import drto
from models.idaes_cstr import DC_START, F_IN, VOLUME, build, scaled_solve, tag_scaling

m = build()
ss = pyo.TransformationFactory("drto.steady_state_simulation").create_using(
    m, controls={m.fs.cstr.control_volume.heat.name: 0.0,
                 m.fs.cstr.inlet.flow_vol.name: F_IN})
scaled_solve(ss)
cvs = ss.fs.cstr.control_volume
for j, ssp in (("NaOH", m.ss_naoh), ("EthylAcetate", m.ss_ea),
               ("SodiumAcetate", m.ss_sa), ("Ethanol", m.ss_etoh)):
    ssp.set_value(pyo.value(cvs.material_holdup["Liq", j]))
for j, sgn in (("NaOH", 1), ("EthylAcetate", 1),
               ("SodiumAcetate", -1), ("Ethanol", -1)):
    m.mat0[j] = pyo.value(cvs.material_holdup["Liq", j]) + sgn * DC_START * VOLUME
for k in m.eng_ss:
    m.eng_ss[k] = pyo.value(cvs.energy_holdup[k])

pyo.TransformationFactory("drto.infinite_horizon").apply_to(m)
tag_scaling(m)
drto.cold_start_dynamic(m, profile="exponential", time_constant=3.0)
pyo.TransformationFactory("drto.dynamic_optimization").apply_to(m)
res = scaled_solve(m, tee=True)
print(res.solver.termination_condition)

********************************************************************************

                    ####    ###   /   # /#   #/  ####  #####
                    #   #  #   # /#   #/ ##  /  #      #
                    ####   #   #/ #   /  # #/#  #      ####
                    #      #   /  #  /#  # /##  #      #
                    #       ##/    #/#   #/  #   ####  #####

********************************************************************************
This program contains POUNCE, a pure-Rust interior-point optimization solver
for nonlinear, conic, and global problems (its NLP core is ported from Ipopt).
Released under the Eclipse Public License (EPL) — drop-in compatible with Ipopt.
         For more information visit https://github.com/jkitchin/pounce
********************************************************************************

This is POUNCE version 0.9.0, running with linear solver FERAL.



Number of nonzeros in equality constraint Jacobian...:     4432
Number of nonzeros in inequality constraint Jacobian.:       20
Number of nonzeros in Lagrangian Hessian.............:      535

Total number of variables............................:     1414
                     variables with only lower bounds:      342
                variables with lower and upper bounds:       92
                     variables with only upper bounds:        0
Total number of equality constraints.................:     1343
Total number of inequality constraints...............:        4
        inequality constraints with only lower bounds:        4
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter      objective   inf_pr   inf_du lg(mu)    ||d|| lg(rg) alpha_du alpha_pr  ls
   0  6.7493968e+03 4.69e+02 9.90e+01   -1.0 0.00e+00      - 0.00e+00 0.00e+00   0
   1  6.5992768e+03 4.63e+02 6.19e+01   -1.0 1.20e+03   -4.0 3.7

   4 -7.6858773e+03 6.39e+02 8.18e+01   -1.0 8.67e+04      - 3.12e-02 4.84e-01f  1
   5  2.2776824e+03 1.22e+02 2.94e+01   -1.0 6.20e+04      - 1.87e-01 1.00e+00h  1
   6  3.3802519e+03 1.08e+01 1.10e+01   -1.0 4.04e+04      - 6.27e-01 1.00e+00h  1
   7  3.5253812e+03 4.34e+00 1.33e+01   -1.0 9.62e+04      - 9.09e-01 1.00e+00h  1
   8  3.5487470e+03 6.77e-02 5.55e-01   -1.0 1.08e+06      - 9.90e-01 1.00e+00h  1
   9  3.5490568e+03 9.03e-05 8.39e-04   -1.0 1.06e+08      - 9.91e-01 1.00e+00f  1
iter      objective   inf_pr   inf_du lg(mu)    ||d|| lg(rg) alpha_du alpha_pr  ls
  10  3.5386356e+03 1.03e-01 1.30e-02   -1.7 7.54e+08      - 1.00e+00 1.00e+00f  1
  11  3.5376767e+03 7.12e-03 9.77e-04   -2.5 6.61e+08      - 1.00e+00 1.00e+00h  1
  12  3.5374672e+03 1.54e-04 2.06e-05   -3.8 2.14e+08      - 1.00e+00 1.00e+00h  1


  13  3.5374536e+03 4.43e-07 5.82e-08   -5.7 4.66e+07      - 1.00e+00 1.00e+00h  1
  14  3.5374534e+03 2.79e-09 8.53e-12   -8.6 5.10e+06      - 1.00e+00 1.00e+00h  1


Number of Iterations....: 14

                                   (scaled)                 (unscaled)
Objective...............:   3.5374534385086093e+02    3.5374534385086090e+03
Dual infeasibility......:   8.5272184278950558e-12    8.5272184278950558e-11
Constraint violation....:   2.7939677238464355e-09    7.7486038208007812e-07
Variable bound violation:   0.0000000000000000e+00    0.0000000000000000e+00
Complementarity.........:   2.5074017838124629e-09    2.5074017838124628e-08
Overall NLP error.......:   2.7939677238464355e-09    7.7486038208007812e-07


Number of objective function evaluations             = 15
Number of objective gradient evaluations             = 15
Number of equality constraint evaluations            = 15
Number of inequality constraint evaluations          = 15
Number of equality constraint Jacob

optimal


## One step later, warm

The loop implements the first move and the state advances one sample;
here the model's own solution at t = h stands in for the measurement.
The hooks take the measured state and the whole solution shifts one
sampling time forward, tail included through
`t = tN + atanh(tau)/gamma`.

In [2]:
# one loop iteration: the first move was implemented and the state
# advanced one sample; the model's own solution at t = h is the ideal
# measurement, written into the feedback hooks
cv = m.fs.cstr.control_volume
h = 1.0
for j in ("NaOH", "EthylAcetate", "SodiumAcetate", "Ethanol"):
    m.mat0[j] = pyo.value(cv.material_holdup[h, "Liq", j])
m.eng0["Liq"] = pyo.value(cv.energy_holdup[h, "Liq"])

print(drto.warm_start_dynamic(m))

drto warm_start_dynamic (the previous solution, one step on)
  shift         : 1 time units
  copied        : 1009 values on aligned points
  interpolated  : 496 values between points
  filled        : 0 values past the end
  tail          : shifted through t = tN + atanh(tau)/gamma


## The second solve

From the shifted start, the same solve again.

In [3]:
res = scaled_solve(m, tee=True)
print(res.solver.termination_condition)

********************************************************************************

                    ####    ###   /   # /#   #/  ####  #####
                    #   #  #   # /#   #/ ##  /  #      #
                    ####   #   #/ #   /  # #/#  #      ####
                    #      #   /  #  /#  # /##  #      #
                    #       ##/    #/#   #/  #   ####  #####

********************************************************************************
This program contains POUNCE, a pure-Rust interior-point optimization solver
for nonlinear, conic, and global problems (its NLP core is ported from Ipopt).
Released under the Eclipse Public License (EPL) — drop-in compatible with Ipopt.
         For more information visit https://github.com/jkitchin/pounce
********************************************************************************

This is POUNCE version 0.9.0, running with linear solver FERAL.



Number of nonzeros in equality constraint Jacobian...:     4432
Number of nonzeros in inequality constraint Jacobian.:       20
Number of nonzeros in Lagrangian Hessian.............:      535

Total number of variables............................:     1414
                     variables with only lower bounds:      342
                variables with lower and upper bounds:       92
                     variables with only upper bounds:        0
Total number of equality constraints.................:     1343
Total number of inequality constraints...............:        4
        inequality constraints with only lower bounds:        4
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter      objective   inf_pr   inf_du lg(mu)    ||d|| lg(rg) alpha_du alpha_pr  ls
   0  1.6556230e+02 4.44e+01 9.90e+01   -1.0 0.00e+00      - 0.00e+00 0.00e+00   0
   1  6.6570486e+01 4.39e+01 1.99e+01   -1.0 4.42e+01   -4.0 7.9

   5  7.7352667e+01 2.44e-09 1.00e-06   -1.0 5.65e+08      - 1.00e+00 1.00e+00f  1
   6  6.4361427e+01 1.52e-01 1.92e-02   -2.5 1.41e+08      - 1.00e+00 1.00e+00f  1
   7  6.5844271e+01 2.65e-04 5.30e-05   -2.5 4.83e+09      - 1.00e+00 1.00e+00h  1
   8  6.5576014e+01 1.37e-04 1.78e-05   -3.8 5.97e+01      - 1.00e+00 1.00e+00h  1
   9  6.5562484e+01 4.41e-07 5.78e-08   -5.7 3.37e+00      - 1.00e+00 1.00e+00h  1
iter      objective   inf_pr   inf_du lg(mu)    ||d|| lg(rg) alpha_du alpha_pr  ls
  10  6.5562304e+01 9.78e-09 8.53e-12   -8.6 4.14e-02      - 1.00e+00 1.00e+00h  1


Number of Iterations....: 10

                                   (scaled)                 (unscaled)
Objective...............:   6.5562303953550103e+00    6.5562303953550099e+01
Dual infeasibility......:   8.5278154980314436e-12    8.5278154980314436e-11
Constraint violation....:   9.7788870334625244e-09    6.8172812461853027e-07
Variable bound violation:   0.0000000000000000e+00    0.0000000000000000e+00
Compleme

optimal


The warm-started solve converges in fewer iterations than a fresh
exponential cold start at the same state (10 against 11 on this run,
with the original problem's solve at 14): the shifted solution is
nearly the answer, every value the shift reads is one the previous
solve determined, and the report's zero fills say the tail covered the
whole problem. That is the entire point of warm starting a receding
horizon.